# LitWBTrainingBasicConvnet notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TXSTCODEPLAYGROUND/CodeCS4337Fall2026/blob/main/LitWBTrainingBasicConvnet/lit_wb_training_basic_convnet_notebook.ipynb)

This notebook trains **LitWBTrainingBasicConvnet**: what the project does and what it teaches is in its [README](README.md). Every project has its own notebook like this one, in its folder.

### Choose your route

| Route | Sections to run | Then run the project with |
| --- | --- | --- |
| **Colab, from notebook cells** | 1 (setup), then 2 and 3 (4 and 5 to load or keep training a model) | Option A1 (`!python -m ...`) or Option B (`main(...)`) |
| **Colab, from Colab's terminal** (Colab Pro) | 1 (setup) in the notebook first | Option A2: the same command in the terminal |
| **Locally, from this notebook** | Skip 1, start at [**2. Run the project**](#2.-Run-the-project-(choose-one-option)) | Option A1 or B, with the `.venv` as the notebook's kernel |
| **Locally, from your terminal** | None, no notebook needed | `python -m LitWBTrainingBasicConvnet --config config01.json` from the repo root, with the `.venv` activated |

In Colab, the setup cells are needed even when you train in the terminal: Google Drive can only be mounted, and Colab Secrets only read, from a notebook cell. Locally, set up the repo once first (`.venv`, requirements, and `.env`, see the main README), and open this notebook from the project folder.

In Colab, run the cells from top to bottom.

1. **Use a GPU:** *Runtime → Change runtime type → GPU* (training also works on CPU, just slower).
2. **Mount Google Drive** so your results survive runtime resets.
3. **Set up the repo:** clone (or update) it, apply the Colab settings, and install only the missing requirements.
4. **Load your API keys** (optional) from Colab Secrets.
5. **Run LitWBTrainingBasicConvnet**, either as a terminal command or from Python, and look at the results.
6. **Load a trained model** later, without training again (section 4).
7. **Keep training** a run for more epochs, or finish one cut off by a disconnect (section 5).

Before a long training run, read the [pro tips](#pro-tips-keep-training-running) at the end.

## 1. Setup

> **Running locally?** Skip this section: these cells only work in Colab. Jump to [**2. Run the project**](#2.-Run-the-project-(choose-one-option)).

Run these cells once per session (and again after the runtime resets): mount Drive, set up the repo, change directory, and load API keys. The setup cell is safe to re-run, e.g. to get new projects with `git pull`; run the API-key cell again after it.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
%%bash
set -e

cd /content
if [ -d CodeCS4337Fall2026/.git ]; then
  git -C CodeCS4337Fall2026 pull --ff-only
else
  git clone https://github.com/TXSTCODEPLAYGROUND/CodeCS4337Fall2026.git
fi
cd CodeCS4337Fall2026

# Colab settings: data on the local disk, results in Google Drive.
cp .envcolab .env

# Install only packages Colab does not already have (versions are not pinned).
grep -vE '^\s*(#|$)' requirements.txt \
  | sed -E 's/[<>=!~;[ ].*//' \
  | while read -r pkg; do
      if pip show "$pkg" > /dev/null 2>&1; then
        echo "skip $pkg (already installed)"
      else
        pip install -q "$pkg" && echo "installed $pkg"
      fi
    done

In [ ]:
# The bash cell above cannot change the notebook's working directory, so do it here.
%cd /content/CodeCS4337Fall2026

### Load API keys (optional)

This project logs every run to Weights & Biases (W&B), which needs `WANDB_API_KEY`. Without it, runs are logged to W&B offline and can be uploaded later with the `wandb sync` command printed at the end of the run.

1. Add the keys as Colab **Secrets**: key icon in the left sidebar → *Add new secret*, named `WANDB_API_KEY`. Never type keys into a notebook cell or into `.envcolab`, which is public on GitHub.
2. Run the cell below **after the setup cells and before running the project**, whether you run it from this notebook (Option A1 or B) or from Colab's terminal (A2). Run it again whenever you re-run the setup cell.

The cell loads `.env`, then looks up every key that is still missing in your Secrets (Colab may ask you to *Grant access*). Keys it finds are added to this notebook's environment and saved to `/content/CodeCS4337Fall2026/.env`, so terminal runs find them too. That file is not in git and is deleted when the runtime resets. Values are never printed.

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
from google.colab import userdata

ENV_FILE = Path("/content/CodeCS4337Fall2026/.env")
KEYS = ["WANDB_API_KEY"]

if load_dotenv(ENV_FILE):
    print(f"Loaded settings from {ENV_FILE}")
else:
    print(f"No settings in {ENV_FILE}: run the setup cells first")

for key in KEYS:
    if os.getenv(key):
        print(f"{key}: found in .env")
        continue
    try:
        os.environ[key] = userdata.get(key)
    except Exception:  # noqa: BLE001 (Secret not added or access not granted)
        print(f"{key}: not found, W&B will log offline")
        continue
    with ENV_FILE.open("a") as env_file:
        env_file.write(f"\n{key}={os.environ[key]}\n")
    print(f"{key}: loaded from Colab Secrets and saved to .env")

## 2. Run the project (choose one option)

Both options run exactly the same training and save results to the same folder (Google Drive in Colab, `runs/` locally). Pass a config name from the project's `configs/` folder. To try your own settings, copy `config01.json` in the file browser (left sidebar, `CodeCS4337Fall2026/LitWBTrainingBasicConvnet/configs/`), e.g. to `config02.json`, edit it, and pass its name.

**Run the next cell first** (in Colab and locally): it finds the repo folder and makes the project importable.

| Option | How | When to use it |
| --- | --- | --- |
| **A. Terminal command** | `!` in a notebook cell, or Colab's terminal (Colab Pro) | Same command as on your own machine |
| **B. From Python** | `from LitWBTrainingBasicConvnet import main` | Simple, everything stays in the notebook |

In [ ]:
import sys
from pathlib import Path

# Colab: the clone made by the setup cell. Locally: the parent of this notebook's project folder.
IN_COLAB = "google.colab" in sys.modules
REPO = Path("/content/CodeCS4337Fall2026") if IN_COLAB else Path.cwd().parent
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print(f"Repo folder: {REPO}")

### Where your results go

Run the cell below and look at what it prints: `OUTPUT_DIR` is the folder where **every run of this project is saved**: checkpoints, metrics, plots, and `runs_summary.csv`. It is set in `.env` at the repo root.

- **In Colab** it is in your Google Drive, `/content/drive/MyDrive/CodeCS4337Fall2026/runs` (from `.envcolab`), so your results survive runtime resets. Open *MyDrive → CodeCS4337Fall2026 → runs* in Google Drive to find them later.
- **Locally** it is `runs`, i.e. the `runs/` folder of the repo (relative paths are relative to the repo root).

Each run gets its own folder, `<OUTPUT_DIR>/LitWBTrainingBasicConvnet/<config>/<timestamp>/`.

In [ ]:
import os

from dotenv import load_dotenv

# Colab: the copy of .envcolab made by the setup cell. Locally: your own .env.
load_dotenv(REPO / ".env")
!echo "OUTPUT_DIR=$OUTPUT_DIR"
output_dir = REPO / os.getenv("OUTPUT_DIR", "runs")
print(f"Runs of this project are saved in: {output_dir / 'LitWBTrainingBasicConvnet'}")

### Option A: Run as a terminal command

**A1. From a notebook cell (everyone):** a line starting with `!` is sent to the shell, and `{REPO}` is replaced by the repo folder found above. Run the cell below. Locally, if it reports a missing package, the shell is not using the `.venv`'s Python: use Option B, or run the same command in your own terminal with the `.venv` activated.

In [ ]:
!cd "{REPO}" && python -m LitWBTrainingBasicConvnet --config config01.json

**A2. From Colab's terminal (Colab Pro):** open the terminal from the **Terminal** button at the bottom left of the Colab window, then run:

```bash
cd /content/CodeCS4337Fall2026
python -m LitWBTrainingBasicConvnet --config config01.json
```

or, from inside the project folder:

```bash
cd /content/CodeCS4337Fall2026/LitWBTrainingBasicConvnet
python main.py --config config01.json
```

The terminal runs on the same machine as this notebook, so it uses the same files, packages, and Drive mount. **You still need to run the setup cells above first**: Drive can only be mounted from a notebook cell, and Colab Secrets can only be read from a notebook cell (the API-key cell saves them to `.env`, where the terminal finds them). While training runs in the terminal, the notebook stays free for other work.

### Option B: Run from Python

The project can be imported because the cell at the start of this section added the repo folder to Python's import path. This works in Colab and locally.

In [ ]:
from LitWBTrainingBasicConvnet import main

main("config01.json")

## 3. Results

Each run is saved under `<OUTPUT_DIR>/LitWBTrainingBasicConvnet/<config>/<timestamp>/`: in Colab that is Google Drive, `MyDrive/CodeCS4337Fall2026/runs/`; locally, the `runs/` folder of the repo (both set by `OUTPUT_DIR` in `.env`). `runs_summary.csv` has one row per run of a config.

**Early stopping.** `config01.json` has `"early_stopping": true` and `"patience": 3` in its `"training"` section: training stops when `val_acc` has not improved for 3 epochs in a row, so `"epochs"` is the **most** epochs a run can take. When a run stops early, it prints after which epoch; the number of epochs it actually ran is the `epochs_run` column of `runs_summary.csv` and in the W&B run's summary. The model tested is still the best checkpoint, not the last epoch. Set `"early_stopping": false` to always train all `"epochs"`.

In [ ]:
import os

from dotenv import load_dotenv

load_dotenv(REPO / ".env")
# OUTPUT_DIR from .env: absolute in Colab (Drive), relative to the repo locally.
runs_dir = REPO / os.getenv("OUTPUT_DIR", "runs")
print((runs_dir / "LitWBTrainingBasicConvnet/config01/runs_summary.csv").read_text())

## 4. Load a trained model

Running a config again trains a **new** model from scratch, unless you resume an earlier run (section 5). To use a model you already trained (after a runtime reset, to test it again, or to try it on images), load it with `load_model`: it finds the checkpoint under `<OUTPUT_DIR>/LitWBTrainingBasicConvnet/`, rebuilds the network with that run's settings, and loads its weights. Nothing is trained.

**Before this section:** you need at least one run of the config, trained in section 2 (in Colab: trained with Drive mounted, so it was saved to Drive). Then, in Colab, run the setup cells of section 1 (Drive must be mounted, that is where your runs are); in Colab and locally, run the first cell of section 2, which finds the repo folder.

| Call | What it loads |
| --- | --- |
| `load_model("config01")` | The newest run of `config01`, its best checkpoint (`best_*.ckpt`) |
| `load_model("config01/2026-10-02_12-29-09")` | One specific run: the folder name of the run, under `LitWBTrainingBasicConvnet/config01/` |
| `load_model("config01", which="last")` | The last epoch instead of the best one |
| `load_model("config01/2026-10-02_12-29-09/best_epoch07_valacc0.9260.ckpt")` | One specific checkpoint file of a run |
| `load_model("config02")` | The newest run of your own `config02.json` |

Each run folder keeps **two checkpoints**: the best epoch (`best_*.ckpt`) and the last epoch (`last.ckpt`); older ones are replaced during training. The model comes back ready for predictions (in eval mode, on the CPU; add `device="cuda"` for the GPU). It prints which checkpoint it loaded.

In [ ]:
from LitWBTrainingBasicConvnet import load_model

model = load_model("config01")

Try the loaded model on the test set: the cell below prints its predictions for the first 8 test images and its accuracy on all 10,000. Images get the same normalization as in training. The accuracy should match the run's `test_acc` in `runs_summary.csv` when you loaded the best checkpoint.

In [ ]:
import os

import torch
from dotenv import load_dotenv
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

from LitWBTrainingBasicConvnet.dataloaders.fashion_mnist import (
    FASHION_MNIST_MEAN,
    FASHION_MNIST_STD,
)

load_dotenv(REPO / ".env")
transform = transforms.Compose(
    [transforms.ToTensor(), transforms.Normalize(FASHION_MNIST_MEAN, FASHION_MNIST_STD)]
)
test_set = datasets.FashionMNIST(
    REPO / os.getenv("DATA_DIR", "data"),
    train=False,
    download=True,
    transform=transform,
)

images, labels = zip(*(test_set[i] for i in range(8)))
with torch.no_grad():
    preds = model(torch.stack(images)).argmax(dim=1)
for pred, label in zip(preds, labels):
    print(f"predicted {test_set.classes[pred]:<12} true {test_set.classes[label]}")

with torch.no_grad():
    correct = sum(
        (model(x).argmax(dim=1) == y).sum().item()
        for x, y in DataLoader(test_set, 1000)
    )
print(f"Test accuracy: {correct / len(test_set):.4f}")

## 5. Keep training a run

`load_model` gives you a trained model to use; it does not train it further. To **continue training** an earlier run, for more epochs or to finish a run cut off by a Colab disconnect, pass `resume_from` to `main`. Training continues from the run's last checkpoint (`last.ckpt`, saved after every epoch) and runs the config's `"epochs"` **more** epochs: a 10-epoch run resumed with `config01.json` ends after epoch 20, or earlier if early stopping stops it.

**Before this section:** the same as for section 4 (a saved run; in Colab, section 1 with Drive mounted; then the first cell of section 2).

| Call | What it continues |
| --- | --- |
| `main("config01.json", resume_from="config01")` | The newest run of `config01`, for `"epochs"` more epochs. Run it again to continue further: the newest run is then the continued one. |
| `main("config01.json", resume_from="config01/2026-10-02_12-29-09")` | One specific run (its folder name) |
| `main("config01.json", resume_from="config01/2026-10-02_12-29-09/last.ckpt")` | One specific checkpoint file |
| `main("config01_more.json", resume_from="config01")` | A different number of extra epochs: a copy of the config with another `"epochs"` (saved under `config01_more/`) |

What comes from where:

- **From the checkpoint:** the weights, the epoch count, and the optimizer state, including its **learning rate** and weight decay. Changing `lr` or `weight_decay` in the config has no effect when resuming.
- **From the config:** the number of extra `"epochs"`, the data settings (e.g. `batch_size`), the seed, and logging. The **model settings must match the run's** (e.g. `dropout`): the saved weights only fit the same network.
- **Early stopping** starts counting again: a run that stopped early can be resumed, and gets the config's `"patience"` epochs to beat the **earlier run's best** `val_acc`.
- The continued run is saved as a **new run folder**, with `resumed_from` in its `hparams.json`, and gets its own row in `runs_summary.csv`. Its best checkpoint and test score are those of the continued run. The earlier run is not changed.

The same from a terminal: `python -m LitWBTrainingBasicConvnet --config config01.json --resume-from config01`.

In [ ]:
from LitWBTrainingBasicConvnet import main

# The config's "epochs" more epochs, on top of the newest config01 run.
main("config01.json", resume_from="config01")

## Pro tips: keep training running

Colab disconnects runtimes that look idle (no interaction for a while, often around 90 minutes) and also caps how long a runtime can run in total (around 12 hours on the free tier, longer on paid plans). The exact limits vary and are not guaranteed. When the runtime disconnects, training stops and everything under `/content/` (outside Drive) is deleted.

**Save your state in Google Drive**
- Always mount Drive *before* training. Results then go straight to Drive (`OUTPUT_DIR` in `.envcolab`), not to the temporary disk.
- Two checkpoints are saved to Drive *during* training: the best model so far (`best_epochXX_valaccY.ckpt`), every time validation accuracy improves, and the last epoch (`last.ckpt`), after every epoch. If the runtime disconnects mid-run, both are kept.
- The run's other files (results, plots, and the row in `runs_summary.csv`) are written only when the run finishes. To finish an interrupted run, resume it (section 5): it continues from the last saved epoch.
- Keep your notebook in Drive too: *File → Save a copy in Drive*. Keep copies of configs you edited in Drive, since `/content/CodeCS4337Fall2026` is deleted on reset.

**Avoid idle disconnects**
- Keep the Colab browser tab open, and don't let your computer go to sleep (plug in your laptop and turn off sleep while training).
- Check on the notebook now and then (scroll, look at the progress bars). Running in the terminal does *not* prevent disconnects: the runtime is shared.
- Don't use scripts or browser extensions that fake activity (auto-clickers); they can get your Colab usage restricted.
- With Colab Pro+, *background execution* keeps the runtime alive even after you close the browser.

**Plan your runs**
- Test a new config with few epochs first (e.g. `"epochs": 1`), then start the full run.
- Prefer several shorter runs over one very long run; each run saves its own results.
- After a disconnect, re-run the setup cells (including the API-key cell), then resume the run with `resume_from` (section 5) instead of starting over.
- When you are done, free the GPU with *Runtime → Disconnect and delete runtime*. It saves your GPU quota (or compute units on paid plans) for next time.